# 2. Model Training: Naive MLP Baseline & InfoNCE Model
This notebook loads the cached visual and audio features, and trains two MLP heads to map (image, audio) $\rightarrow$ teacher video embedding.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np

# Load source code modules (we will clone the local repo inside Kaggle or load them directly)
import sys
sys.path.append('.')
from src.models import MLPApproximator
from src.loss import InfoNCELoss
from src.dataset import MultimodalEmbeddingDataset
from src.metrics import calculate_proximity, calculate_retrieval_metrics

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cpu


## Step 1: Load Datasets


In [2]:
import os

# Robust path resolution for local and remote (Kaggle) directories
features_dir = None
for candidate in ["features/patched_features", "../features/patched_features", "patched_features"]:
    if os.path.exists(os.path.join(candidate, "train_features.pt")):
        features_dir = candidate
        break

if features_dir is None:
    raise FileNotFoundError("Could not find train_features.pt in any standard path")

train_path = os.path.join(features_dir, "train_features.pt")
test_path = os.path.join(features_dir, "test_features.pt")

train_dataset = MultimodalEmbeddingDataset(file_path=train_path)
test_dataset = MultimodalEmbeddingDataset(file_path=test_path)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Loaded {len(train_dataset)} training items, {len(test_dataset)} test items.")


Loaded 7010 training items, 1000 test items.


## Step 2: Train Baseline 1 (MLP + Cosine Similarity Loss)
This serves as the baseline: concatenate image (512) and audio (128) embeddings and project them to teacher dimension (1024) using a simple MLP trained with cosine similarity loss.


In [3]:
model_baseline = MLPApproximator(input_dim=640, hidden_dims=[512, 1024], output_dim=1024).to(device)
optimizer = optim.AdamW(model_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = lambda pred, target: (1 - torch.nn.functional.cosine_similarity(pred, target)).mean()

epochs = 30
checkpoint_pcts = [0.25, 0.50, 0.75]
checkpoint_epochs = [int(epochs * pct) for pct in checkpoint_pcts]

print(f"Training Baseline 1. Will save checkpoints to models/ at epochs: {checkpoint_epochs}")

for epoch in range(epochs):
    model_baseline.train()
    epoch_loss = 0.0
    for batch in train_loader:
        z_img = batch['z_img'].to(device)
        z_aud = batch['z_aud'].to(device)
        v_teacher = batch['v_teacher'].to(device)
        
        optimizer.zero_grad()
        v_pred = model_baseline(z_img, z_aud)
        loss = criterion(v_pred, v_teacher)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * z_img.size(0)
        
    train_loss = epoch_loss / len(train_dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Cosine Loss: {train_loss:.4f}")
        
    # Intermediate checkpoint saving to models/
    if (epoch + 1) in checkpoint_epochs:
        pct = int(((epoch + 1) / epochs) * 100)
        chk_path = f"models/mlp_cosine_{pct}pct.pt"
        os.makedirs("models", exist_ok=True)
        torch.save(model_baseline.state_dict(), chk_path)
        print(f"Intermediate checkpoint saved: {chk_path} at epoch {epoch+1}")

# Save the final model weights to models/
os.makedirs("models", exist_ok=True)
torch.save(model_baseline.state_dict(), "models/mlp_cosine.pt")
print("Baseline 1 training complete and weights saved to models/mlp_cosine.pt!")


Training Baseline 1. Will save checkpoints to models/ at epochs: [7, 15, 22]


Epoch 01/30 | Train Cosine Loss: 0.4186


Epoch 05/30 | Train Cosine Loss: 0.2680


Intermediate checkpoint saved: models/mlp_cosine_23pct.pt at epoch 7


Epoch 10/30 | Train Cosine Loss: 0.2244


Epoch 15/30 | Train Cosine Loss: 0.2001
Intermediate checkpoint saved: models/mlp_cosine_50pct.pt at epoch 15


Epoch 20/30 | Train Cosine Loss: 0.1840


Intermediate checkpoint saved: models/mlp_cosine_73pct.pt at epoch 22


Epoch 25/30 | Train Cosine Loss: 0.1718


Epoch 30/30 | Train Cosine Loss: 0.1637
Baseline 1 training complete and weights saved to models/mlp_cosine.pt!


## Step 3: Train Method A (MLP + InfoNCE Loss)
Instead of minimizing cosine distance independently, we optimize alignment over batches using symmetric InfoNCE contrastive loss.


In [4]:
model_infonce = MLPApproximator(input_dim=640, hidden_dims=[512, 1024], output_dim=1024).to(device)
optimizer = optim.AdamW(model_infonce.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = InfoNCELoss(temperature=0.07, symmetric=True)

epochs = 30
checkpoint_pcts = [0.25, 0.50, 0.75]
checkpoint_epochs = [int(epochs * pct) for pct in checkpoint_pcts]

print(f"Training Method A (InfoNCE). Will save checkpoints to models/ at epochs: {checkpoint_epochs}")

for epoch in range(epochs):
    model_infonce.train()
    epoch_loss = 0.0
    for batch in train_loader:
        z_img = batch['z_img'].to(device)
        z_aud = batch['z_aud'].to(device)
        v_teacher = batch['v_teacher'].to(device)
        
        optimizer.zero_grad()
        v_pred = model_infonce(z_img, z_aud)
        loss = criterion(v_pred, v_teacher)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * z_img.size(0)
        
    train_loss = epoch_loss / len(train_dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train InfoNCE Loss: {train_loss:.4f}")
        
    # Intermediate checkpoint saving to models/
    if (epoch + 1) in checkpoint_epochs:
        pct = int(((epoch + 1) / epochs) * 100)
        chk_path = f"models/mlp_infonce_{pct}pct.pt"
        os.makedirs("models", exist_ok=True)
        torch.save(model_infonce.state_dict(), chk_path)
        print(f"Intermediate checkpoint saved: {chk_path} at epoch {epoch+1}")

# Save the final model weights to models/
os.makedirs("models", exist_ok=True)
torch.save(model_infonce.state_dict(), "models/mlp_infonce.pt")
print("Method A (InfoNCE) training complete and weights saved to models/mlp_infonce.pt!")


Training Method A (InfoNCE). Will save checkpoints to models/ at epochs: [7, 15, 22]


Epoch 01/30 | Train InfoNCE Loss: 3.4615


Epoch 05/30 | Train InfoNCE Loss: 1.6676


Intermediate checkpoint saved: models/mlp_infonce_23pct.pt at epoch 7


Epoch 10/30 | Train InfoNCE Loss: 1.0419


Epoch 15/30 | Train InfoNCE Loss: 0.7357
Intermediate checkpoint saved: models/mlp_infonce_50pct.pt at epoch 15


Epoch 20/30 | Train InfoNCE Loss: 0.6251


Intermediate checkpoint saved: models/mlp_infonce_73pct.pt at epoch 22


Epoch 25/30 | Train InfoNCE Loss: 0.5219


Epoch 30/30 | Train InfoNCE Loss: 0.4945
Method A (InfoNCE) training complete and weights saved to models/mlp_infonce.pt!
